<a href="https://colab.research.google.com/github/uvindu827/J26-DS-350/blob/Indhi_Component2_CHL_Forecast/31_9P_16T_3D_Spatial_Temporal_CHL_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================================================
# STEP 31.9P-16T-17
# 3D SPATIAL-TEMPORAL CHL VISUALIZATION
# ENVIRONMENT & DATA VERIFICATION
# ================================================================

import os
import json
import numpy as np
import pandas as pd

print("=" * 70)
print("STEP 31.9P-16T-17")
print("3D SPATIAL-TEMPORAL CHL VISUALIZATION")
print("ENVIRONMENT & DATA VERIFICATION")
print("=" * 70)


# ================================================================
# PROJECT PATHS
# ================================================================

PROJECT_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "06_models",
    "02_XGBoost",
    "hyperparameter_optimization",
    "final_tuned_candidate"
)

FINAL_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "xgboost_final_tuned_candidate.json"
)

FINAL_CONFIG_PATH = os.path.join(
    MODEL_DIR,
    "xgboost_final_tuned_candidate_configuration.json"
)

FEATURE_LIST_PATH = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_feature_columns.csv"
)

TEST_DATA_PATH = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_test.csv"
)

TUNED_TEST_PREDICTIONS_PATH = os.path.join(
    MODEL_DIR,
    "final_test_evaluation",
    "xgboost_final_tuned_test_predictions.csv"
)

SHAP_DIR = os.path.join(
    MODEL_DIR,
    "shap_explainability"
)

OUTPUT_DIR = os.path.join(
    MODEL_DIR,
    "3d_spatial_temporal_visualization"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


print("\n" + "=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print(f"\nProject:")
print(PROJECT_DIR)

print(f"\nFinal tuned model:")
print(FINAL_MODEL_PATH)

print(f"\nFinal feature list:")
print(FEATURE_LIST_PATH)

print(f"\nFinal test dataset:")
print(TEST_DATA_PATH)

print(f"\nSaved tuned test predictions:")
print(TUNED_TEST_PREDICTIONS_PATH)

print(f"\nSHAP directory:")
print(SHAP_DIR)

print(f"\n3D visualization output directory:")
print(OUTPUT_DIR)


# ================================================================
# ENVIRONMENT CHECK
# ================================================================

print("\n" + "=" * 70)
print("VISUALIZATION ENVIRONMENT CHECK")
print("=" * 70)

try:
    import plotly
    print(f"Plotly version: {plotly.__version__}")
    PLOTLY_AVAILABLE = True
except Exception as e:
    print("Plotly is not available.")
    print(f"Error: {e}")
    PLOTLY_AVAILABLE = False

try:
    import matplotlib
    print(f"Matplotlib version: {matplotlib.__version__}")
    MATPLOTLIB_AVAILABLE = True
except Exception as e:
    print("Matplotlib is not available.")
    print(f"Error: {e}")
    MATPLOTLIB_AVAILABLE = False

try:
    import sklearn
    print(f"Scikit-learn version: {sklearn.__version__}")
    SKLEARN_AVAILABLE = True
except Exception as e:
    print("Scikit-learn is not available.")
    print(f"Error: {e}")
    SKLEARN_AVAILABLE = False


if not PLOTLY_AVAILABLE:
    raise ImportError(
        "Plotly is required for the interactive 3D visualization."
    )

print("\n✓ Visualization environment verified.")


# ================================================================
# REQUIRED FILE CHECK
# ================================================================

print("\n" + "=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

required_files = {
    "Final tuned XGBoost": FINAL_MODEL_PATH,
    "Final configuration": FINAL_CONFIG_PATH,
    "Final feature list": FEATURE_LIST_PATH,
    "Final test dataset": TEST_DATA_PATH,
    "Tuned test predictions": TUNED_TEST_PREDICTIONS_PATH,
}

all_files_available = True

for name, path in required_files.items():
    exists = os.path.isfile(path)

    print(f"{name:<30}: {exists}")

    if not exists:
        all_files_available = False
        print(f"  Missing: {path}")

if not all_files_available:
    raise FileNotFoundError(
        "One or more required visualization files are missing."
    )

print("\n✓ All required visualization files found.")


# ================================================================
# LOAD FINAL FEATURE LIST
# ================================================================

print("\n" + "=" * 70)
print("LOADING FINAL FEATURE LIST")
print("=" * 70)

feature_df = pd.read_csv(FEATURE_LIST_PATH)

print(f"Feature list shape: {feature_df.shape}")

# Identify feature-name column safely
possible_feature_columns = [
    "feature",
    "feature_name",
    "Feature",
    "Feature_Name"
]

feature_column = None

for col in possible_feature_columns:
    if col in feature_df.columns:
        feature_column = col
        break

if feature_column is None:
    # Fall back to the first column if the file has only one logical
    # feature-name column.
    if feature_df.shape[1] >= 1:
        feature_column = feature_df.columns[0]
    else:
        raise ValueError("Could not identify the feature-name column.")

FINAL_FEATURES = feature_df[feature_column].astype(str).tolist()

print(f"Number of final features: {len(FINAL_FEATURES)}")

if len(FINAL_FEATURES) != 69:
    raise ValueError(
        f"Expected exactly 69 final model features, "
        f"found {len(FINAL_FEATURES)}."
    )

if len(set(FINAL_FEATURES)) != 69:
    raise ValueError("Duplicate features found in final feature list.")

print("✓ Exactly 69 final model features confirmed.")


# ================================================================
# LOAD TEST DATA
# ================================================================

print("\n" + "=" * 70)
print("LOADING FINAL TEST DATA FOR VISUALIZATION VERIFICATION")
print("=" * 70)

test_df = pd.read_csv(TEST_DATA_PATH)

print(f"Test dataset shape: {test_df.shape}")
print(f"Test observations: {len(test_df):,}")
print(f"Test columns: {len(test_df.columns)}")


# ================================================================
# TEST COLUMN CHECK
# ================================================================

print("\n" + "=" * 70)
print("SPATIAL-TEMPORAL COLUMN CHECK")
print("=" * 70)

required_spatial_temporal_columns = [
    "date",
    "latitude",
    "longitude",
    "chlorophyll_a"
]

missing_spatial_temporal = [
    col for col in required_spatial_temporal_columns
    if col not in test_df.columns
]

print(f"Missing spatial-temporal columns: {len(missing_spatial_temporal)}")

if missing_spatial_temporal:
    print("Missing columns:")
    for col in missing_spatial_temporal:
        print(f"  - {col}")

    raise ValueError(
        "Required spatial-temporal columns are missing."
    )

print("✓ Date, latitude, longitude and actual CHL are available.")


# ================================================================
# DATE VERIFICATION
# ================================================================

print("\n" + "=" * 70)
print("DATE / TIME VERIFICATION")
print("=" * 70)

test_df["date"] = pd.to_datetime(
    test_df["date"],
    errors="coerce"
)

date_nan = test_df["date"].isna().sum()

print(f"Invalid date values: {date_nan}")

if date_nan > 0:
    raise ValueError(
        "Invalid or missing date values detected."
    )

print(f"Minimum date: {test_df['date'].min()}")
print(f"Maximum date: {test_df['date'].max()}")
print(f"Unique dates: {test_df['date'].nunique():,}")

print("✓ Date values verified.")


# ================================================================
# SPATIAL VERIFICATION
# ================================================================

print("\n" + "=" * 70)
print("SPATIAL DATA VERIFICATION")
print("=" * 70)

latitude_nan = test_df["latitude"].isna().sum()
longitude_nan = test_df["longitude"].isna().sum()

latitude_inf = np.isinf(test_df["latitude"]).sum()
longitude_inf = np.isinf(test_df["longitude"]).sum()

print(f"Latitude NaN       : {latitude_nan}")
print(f"Longitude NaN      : {longitude_nan}")
print(f"Latitude infinite  : {latitude_inf}")
print(f"Longitude infinite : {longitude_inf}")

if (
    latitude_nan > 0
    or longitude_nan > 0
    or latitude_inf > 0
    or longitude_inf > 0
):
    raise ValueError(
        "Invalid spatial coordinates detected."
    )

print(f"Latitude range : {test_df['latitude'].min()} "
      f"to {test_df['latitude'].max()}")

print(f"Longitude range: {test_df['longitude'].min()} "
      f"to {test_df['longitude'].max()}")

print(f"Unique latitude locations : {test_df['latitude'].nunique():,}")
print(f"Unique longitude locations: {test_df['longitude'].nunique():,}")

print("✓ Spatial coordinates verified.")


# ================================================================
# ACTUAL CHL VERIFICATION
# ================================================================

print("\n" + "=" * 70)
print("ACTUAL CHL VERIFICATION")
print("=" * 70)

chl_nan = test_df["chlorophyll_a"].isna().sum()
chl_inf = np.isinf(test_df["chlorophyll_a"]).sum()

print(f"CHL NaN      : {chl_nan}")
print(f"CHL infinite : {chl_inf}")

if chl_nan > 0 or chl_inf > 0:
    raise ValueError(
        "Invalid actual CHL values detected."
    )

print(f"Minimum CHL: {test_df['chlorophyll_a'].min():.6f} mg/m³")
print(f"Maximum CHL: {test_df['chlorophyll_a'].max():.6f} mg/m³")
print(f"Mean CHL   : {test_df['chlorophyll_a'].mean():.6f} mg/m³")

print("✓ Actual CHL values verified.")


# ================================================================
# LOAD SAVED TUNED TEST PREDICTIONS
# ================================================================

print("\n" + "=" * 70)
print("LOADING SAVED TUNED XGBOOST TEST PREDICTIONS")
print("=" * 70)

prediction_df = pd.read_csv(
    TUNED_TEST_PREDICTIONS_PATH
)

print(f"Prediction dataset shape: {prediction_df.shape}")
print(f"Prediction observations: {len(prediction_df):,}")

print("\nPrediction columns:")
print(prediction_df.columns.tolist())


# ================================================================
# PREDICTION COLUMN CHECK
# ================================================================

print("\n" + "=" * 70)
print("PREDICTION COLUMN CHECK")
print("=" * 70)

required_prediction_columns = [
    "date",
    "latitude",
    "longitude",
    "actual_chlorophyll_a",
    "predicted_chlorophyll_a"
]

missing_prediction_columns = [
    col for col in required_prediction_columns
    if col not in prediction_df.columns
]

print(
    f"Missing required prediction columns: "
    f"{len(missing_prediction_columns)}"
)

if missing_prediction_columns:
    print("Missing:")
    for col in missing_prediction_columns:
        print(f"  - {col}")

    raise ValueError(
        "Required prediction columns are missing."
    )

print("✓ Required spatial-temporal prediction columns found.")


# ================================================================
# PREDICTION DATE VERIFICATION
# ================================================================

prediction_df["date"] = pd.to_datetime(
    prediction_df["date"],
    errors="coerce"
)

if prediction_df["date"].isna().sum() > 0:
    raise ValueError(
        "Invalid prediction dates detected."
    )

print(
    f"Prediction date range: "
    f"{prediction_df['date'].min()} "
    f"to "
    f"{prediction_df['date'].max()}"
)


# ================================================================
# PREDICTION SAFETY CHECK
# ================================================================

print("\n" + "=" * 70)
print("PREDICTION SAFETY CHECK")
print("=" * 70)

predicted_chl_nan = prediction_df[
    "predicted_chlorophyll_a"
].isna().sum()

predicted_chl_inf = np.isinf(
    prediction_df["predicted_chlorophyll_a"]
).sum()

actual_pred_chl_nan = prediction_df[
    "actual_chlorophyll_a"
].isna().sum()

print(f"Predicted CHL NaN      : {predicted_chl_nan}")
print(f"Predicted CHL infinite : {predicted_chl_inf}")
print(f"Actual CHL NaN         : {actual_pred_chl_nan}")

if (
    predicted_chl_nan > 0
    or predicted_chl_inf > 0
    or actual_pred_chl_nan > 0
):
    raise ValueError(
        "Invalid saved prediction values detected."
    )

print("✓ Saved tuned predictions passed safety checks.")


# ================================================================
# ROW COUNT CONSISTENCY CHECK
# ================================================================

print("\n" + "=" * 70)
print("ROW COUNT CONSISTENCY CHECK")
print("=" * 70)

print(f"Test dataset rows       : {len(test_df):,}")
print(f"Prediction dataset rows : {len(prediction_df):,}")

if len(test_df) != len(prediction_df):
    print(
        "⚠ Test and prediction row counts differ. "
        "This will require explicit spatial-temporal alignment."
    )
else:
    print("✓ Test and prediction row counts match.")


# ================================================================
# SPATIAL-TEMPORAL ALIGNMENT CHECK
# ================================================================

print("\n" + "=" * 70)
print("SPATIAL-TEMPORAL ALIGNMENT CHECK")
print("=" * 70)

test_keys = test_df[
    ["date", "latitude", "longitude"]
].copy()

prediction_keys = prediction_df[
    ["date", "latitude", "longitude"]
].copy()

test_keys = test_keys.sort_values(
    ["date", "latitude", "longitude"]
).reset_index(drop=True)

prediction_keys = prediction_keys.sort_values(
    ["date", "latitude", "longitude"]
).reset_index(drop=True)

if test_keys.equals(prediction_keys):
    print("✓ Date + latitude + longitude alignment confirmed.")
else:
    print(
        "⚠ Direct row-wise spatial-temporal alignment "
        "was not confirmed."
    )
    print(
        "The later visualization step will use an explicit "
        "key-based merge rather than assuming row order."
    )


# ================================================================
# VISUALIZATION DATA SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("3D VISUALIZATION DATA SUMMARY")
print("=" * 70)

print(f"Observations available : {len(prediction_df):,}")
print(
    f"Date range             : "
    f"{prediction_df['date'].min().date()} "
    f"→ "
    f"{prediction_df['date'].max().date()}"
)

print(
    f"Latitude range         : "
    f"{prediction_df['latitude'].min()} "
    f"→ "
    f"{prediction_df['latitude'].max()}"
)

print(
    f"Longitude range        : "
    f"{prediction_df['longitude'].min()} "
    f"→ "
    f"{prediction_df['longitude'].max()}"
)

print(
    f"Predicted CHL range    : "
    f"{prediction_df['predicted_chlorophyll_a'].min():.6f} "
    f"→ "
    f"{prediction_df['predicted_chlorophyll_a'].max():.6f} mg/m³"
)

print(
    f"Actual CHL range       : "
    f"{prediction_df['actual_chlorophyll_a'].min():.6f} "
    f"→ "
    f"{prediction_df['actual_chlorophyll_a'].max():.6f} mg/m³"
)


# ================================================================
# FUTURE VISUALIZATION ARCHITECTURE CHECK
# ================================================================

print("\n" + "=" * 70)
print("FUTURE 3D VISUALIZATION ARCHITECTURE")
print("=" * 70)

print("""
Historical CHL
      ↓
Current/API CHL
      ↓
Day +1 predicted CHL
      ↓
Day +2 predicted CHL
      ↓
Day +3 predicted CHL
      ↓
Interactive time dimension
      ↓
Latitude × Longitude × CHL
      ↓
Spatial-temporal CHL change visualization
""")

print("✓ Current dataset structure can support the historical/current")
print("  visualization stage.")
print("✓ Future API predictions will be integrated in later steps.")
print("✓ No future predictions are generated in this step.")


# ================================================================
# SAFETY CHECK
# ================================================================

print("\n" + "=" * 70)
print("FINAL 31.9P-16T-17 SAFETY CHECK")
print("=" * 70)

print("Final XGBoost retrained       : FALSE")
print("Hyperparameters changed       : FALSE")
print("Final model modified          : FALSE")
print("Test predictions regenerated  : FALSE")
print("SHAP recalculated             : FALSE")
print("API data requested            : FALSE")
print("Future CHL predicted          : FALSE")
print("Historical/test data used     : TRUE")
print("Purpose                       : 3D visualization verification")

print("\n✓ Final XGBoost remains untouched.")
print("✓ Existing tuned predictions were only read.")
print("✓ Spatial-temporal data verified.")
print("✓ Visualization environment verified.")
print("✓ Ready for 3D visualization construction.")

print("\n" + "=" * 70)
print("STEP 31.9P-16T-17 COMPLETE")
print("=" * 70)

print("\nNEXT:")
print("31.9P-16T-18 — Historical/Current CHL 3D Visualization")

STEP 31.9P-16T-17
3D SPATIAL-TEMPORAL CHL VISUALIZATION
ENVIRONMENT & DATA VERIFICATION

PROJECT PATHS

Project:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Final tuned model:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/final_tuned_candidate/xgboost_final_tuned_candidate.json

Final feature list:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_feature_columns.csv

Final test dataset:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_test.csv

Saved tuned test predictions:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/final_tuned_candidate/final_test_evaluation/xgboost_final_tuned_test_predictions.csv

SHAP directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/final_tuned_candidate/shap_explainability

3D visualization output directory:
/cont